<a href="https://colab.research.google.com/github/samateh10/football-dw-project/blob/main/Football_DW_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Text cell

# Football Player Performance — Data Warehouse Project

## Phase 2 — Data Management

This notebook covers the complete data pipeline for the football
player performance dataset. We assess data quality, clean the data,
populate the reconciled database, and run the ETL process to load
the data warehouse.

Dataset source: Kaggle — Football Player Stats across top European
and world leagues from 2016 to 2020.

In [52]:
# import all libraries needed
import pandas as pd
import numpy as np

In [53]:
# load the raw dataset
df = pd.read_csv('Data.csv')
print('Shape:', df.shape)

Shape: (660, 15)


## Step 1 — Initial Data Inspection

In [54]:
# view the first 5 rows
df.head()

,Country,League,Club,Player Names,Matches_Played,Substitution,Mins,Goals,xG,xG Per Avg Match,Shots,OnTarget,Shots Per Avg Match,On Target Per Avg Match,Year
0,Spain,La Liga,(BET),Juanmi Callejon,19,16,1849,11,6.62,0.34,48,20,2.47,1.03,2016
1,Spain,La Liga,(BAR),Antoine Griezmann,36,0,3129,16,11.86,0.36,88,41,2.67,1.24,2016
2,Spain,La Liga,(ATL),Luis Suarez,34,1,2940,28,23.21,0.75,120,57,3.88,1.84,2016
3,Spain,La Liga,(CAR),Ruben Castro,32,3,2842,13,14.06,0.47,117,42,3.91,1.40,2016
4,Spain,La Liga,(VAL),Kevin Gameiro,21,10,1745,13,10.65,0.58,50,23,2.72,1.25,2016


In [55]:
# check all column names
print(df.columns.tolist())

['Country', 'League', 'Club', 'Player Names', 'Matches_Played', 'Substitution ', 'Mins', 'Goals', 'xG', 'xG Per Avg Match', 'Shots', 'OnTarget', 'Shots Per Avg Match', 'On Target Per Avg Match', 'Year']


In [56]:
# check data types of each column
print(df.dtypes)

Country                     object
League                      object
Club                        object
Player Names                object
Matches_Played               int64
Substitution                 int64
Mins                         int64
Goals                        int64
xG                         float64
xG Per Avg Match           float64
Shots                        int64
OnTarget                     int64
Shots Per Avg Match        float64
On Target Per Avg Match    float64
Year                         int64
dtype: object


## Step 2 — Data Quality Assessment

In [57]:
# check for missing values in every column
print(df.isnull().sum())

Country                     0
League                      0
Club                       34
Player Names                0
Matches_Played              0
Substitution                0
Mins                        0
Goals                       0
xG                          0
xG Per Avg Match            0
Shots                       0
OnTarget                    0
Shots Per Avg Match         0
On Target Per Avg Match     0
Year                        0
dtype: int64


In [58]:
# check percentage of missing values
print((df.isnull().sum() / len(df)) * 100)

Country                    0.000000
League                     0.000000
Club                       5.151515
Player Names               0.000000
Matches_Played             0.000000
Substitution               0.000000
Mins                       0.000000
Goals                      0.000000
xG                         0.000000
xG Per Avg Match           0.000000
Shots                      0.000000
OnTarget                   0.000000
Shots Per Avg Match        0.000000
On Target Per Avg Match    0.000000
Year                       0.000000
dtype: float64


In [59]:
# check for duplicate rows
print(df.duplicated().sum())

0


In [60]:
# check unique country values
print(df['Country'].unique())

['Spain' 'Italy' 'Germany' 'England' 'Brazil' 'France' 'USA' 'Portugal '
 ' Netherlands']


In [61]:
# check unique league values
print(df['League'].unique())

['La Liga' 'Serie A' 'Bundesliga' 'Premier League'
 'Campeonato Brasileiro SÃ©rie A' 'France Ligue 11' 'France Ligue 20'
 'France Ligue 2' 'France Ligue 12' 'France Ligue 9' 'France Ligue 15'
 'France Ligue 6' 'France Ligue 3' 'France Ligue 16' 'France Ligue 14'
 'France Ligue 4' 'France Ligue 1' 'France Ligue 10' 'France Ligue 7'
 'France Ligue 13' 'France Ligue 8' 'France Ligue 5' 'France Ligue 19'
 'France Ligue 18' 'France Ligue 17' 'MLS' 'Primeira Liga' 'Eredivisie']


## Step 3 — Data Cleaning

In [62]:
# fix all column names
df.columns = [
    'country', 'league', 'club', 'player_name',
    'matches_played', 'substitutions', 'minutes',
    'goals', 'xg', 'xg_per_avg_match', 'shots',
    'on_target', 'shots_per_avg_match',
    'on_target_per_avg_match', 'year'
]
print(df.columns.tolist())

['country', 'league', 'club', 'player_name', 'matches_played', 'substitutions', 'minutes', 'goals', 'xg', 'xg_per_avg_match', 'shots', 'on_target', 'shots_per_avg_match', 'on_target_per_avg_match', 'year']


In [63]:
# fix leading and trailing spaces in country names
df['country'] = df['country'].str.strip()
print(df['country'].unique())

['Spain' 'Italy' 'Germany' 'England' 'Brazil' 'France' 'USA' 'Portugal'
 'Netherlands']


In [64]:
# fix corrupted French league names
df['league'] = df['league'].apply(
    lambda x: 'Ligue 1' if str(x).startswith('France Ligue') else x
)
print(df['league'].unique())

['La Liga' 'Serie A' 'Bundesliga' 'Premier League'
 'Campeonato Brasileiro SÃ©rie A' 'Ligue 1' 'MLS' 'Primeira Liga'
 'Eredivisie']


In [65]:
# fix encoding error in Brazilian league name
df['league'] = df['league'].str.replace(
    'Campeonato Brasileiro SÃ©rie A',
    'Campeonato Brasileiro Série A'
)
print(df['league'].unique())

['La Liga' 'Serie A' 'Bundesliga' 'Premier League'
 'Campeonato Brasileiro Série A' 'Ligue 1' 'MLS' 'Primeira Liga'
 'Eredivisie']


In [66]:
# fill missing club values with Unknown
df['club'] = df['club'].fillna('Unknown')
print(df.isnull().sum())

country                    0
league                     0
club                       0
player_name                0
matches_played             0
substitutions              0
minutes                    0
goals                      0
xg                         0
xg_per_avg_match           0
shots                      0
on_target                  0
shots_per_avg_match        0
on_target_per_avg_match    0
year                       0
dtype: int64


In [67]:
# save clean data to csv
df.to_csv('clean_data.csv', index=False)
print('Clean data saved — rows:', len(df))

Clean data saved — rows: 660


## Step 4 — Populating the Reconciled Database

In [68]:
# import sqlite3 — no installation needed it is built into Python
import sqlite3

# create the database file
conn = sqlite3.connect('football_dw.db')

# create cursor
cursor = conn.cursor()

print('Connected to SQLite database successfully')

Connected to SQLite database successfully


In [69]:
# SQLite uses ? as placeholder instead of %s
# this function converts all %s to ? automatically
import re

def fix_query(query):
    return query.replace('%s', '?')

In [70]:
# create Country table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Country (
        country_id INTEGER PRIMARY KEY AUTOINCREMENT,
        country_name TEXT NOT NULL UNIQUE
    )
''')
conn.commit()
print('Country table ready')

Country table ready


In [71]:
# create League table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS League (
        league_id INTEGER PRIMARY KEY AUTOINCREMENT,
        league_name TEXT NOT NULL,
        country_id INTEGER NOT NULL,
        FOREIGN KEY (country_id) REFERENCES Country(country_id)
    )
''')
conn.commit()
print('League table ready')

League table ready


In [72]:
# create Club table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Club (
        club_id INTEGER PRIMARY KEY AUTOINCREMENT,
        club_code TEXT,
        league_id INTEGER NOT NULL,
        FOREIGN KEY (league_id) REFERENCES League(league_id)
    )
''')
conn.commit()
print('Club table ready')

Club table ready


In [73]:
# create Player table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Player (
        player_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player_name TEXT NOT NULL
    )
''')
conn.commit()
print('Player table ready')

Player table ready


In [74]:
# create Performance table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Performance (
        performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player_id INTEGER NOT NULL,
        club_id INTEGER NOT NULL,
        year INTEGER NOT NULL,
        matches_played INTEGER,
        substitutions INTEGER,
        minutes INTEGER,
        goals INTEGER,
        xg REAL,
        xg_per_avg_match REAL,
        shots INTEGER,
        on_target INTEGER,
        shots_per_avg_match REAL,
        on_target_per_avg_match REAL,
        FOREIGN KEY (player_id) REFERENCES Player(player_id),
        FOREIGN KEY (club_id) REFERENCES Club(club_id)
    )
''')
conn.commit()
print('Performance table ready')

Performance table ready


In [98]:
# clear all tables before inserting to avoid duplicates
cursor.execute('DELETE FROM fact_performance')
cursor.execute('DELETE FROM dim_player')
cursor.execute('DELETE FROM dim_club')
cursor.execute('DELETE FROM dim_year')
cursor.execute('DELETE FROM Performance')
cursor.execute('DELETE FROM Club')
cursor.execute('DELETE FROM League')
cursor.execute('DELETE FROM Player')
cursor.execute('DELETE FROM Country')
conn.commit()
print('Tables cleared and ready for fresh insert')

Tables cleared and ready for fresh insert


In [99]:
# insert unique countries
countries = df['country'].unique()

for country in countries:
    cursor.execute(
        'INSERT OR IGNORE INTO Country (country_name) VALUES (?)',
        (country,)
    )

conn.commit()
print('Countries inserted:', len(countries))

Countries inserted: 9


In [76]:
# insert unique leagues
leagues = df[['league', 'country']].drop_duplicates()

for _, row in leagues.iterrows():
    cursor.execute(
        'SELECT country_id FROM Country WHERE country_name = ?',
        (row['country'],)
    )
    country_id = cursor.fetchone()[0]

    cursor.execute(
        'INSERT OR IGNORE INTO League (league_name, country_id) VALUES (?, ?)',
        (row['league'], country_id)
    )

conn.commit()
print('Leagues inserted:', len(leagues))

Leagues inserted: 9


In [77]:
# insert unique clubs
clubs = df[['club', 'league']].drop_duplicates()

for _, row in clubs.iterrows():
    cursor.execute(
        'SELECT league_id FROM League WHERE league_name = ?',
        (row['league'],)
    )
    league_id = cursor.fetchone()[0]

    cursor.execute(
        'INSERT OR IGNORE INTO Club (club_code, league_id) VALUES (?, ?)',
        (row['club'], league_id)
    )

conn.commit()
print('Clubs inserted:', len(clubs))

Clubs inserted: 240


In [78]:
# insert unique players
players = df['player_name'].unique()

for player in players:
    cursor.execute(
        'INSERT OR IGNORE INTO Player (player_name) VALUES (?)',
        (player,)
    )

conn.commit()
print('Players inserted:', len(players))

Players inserted: 444


In [79]:
# insert all performance records
for _, row in df.iterrows():
    cursor.execute(
        'SELECT player_id FROM Player WHERE player_name = ?',
        (row['player_name'],)
    )
    player_id = cursor.fetchone()[0]

    cursor.execute(
        'SELECT club_id FROM Club WHERE club_code = ?',
        (row['club'],)
    )
    club_id = cursor.fetchone()[0]

    cursor.execute('''
        INSERT INTO Performance (
            player_id, club_id, year, matches_played,
            substitutions, minutes, goals, xg,
            xg_per_avg_match, shots, on_target,
            shots_per_avg_match, on_target_per_avg_match
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        player_id, club_id, row['year'],
        row['matches_played'], row['substitutions'],
        row['minutes'], row['goals'], row['xg'],
        row['xg_per_avg_match'], row['shots'],
        row['on_target'], row['shots_per_avg_match'],
        row['on_target_per_avg_match']
    ))

conn.commit()
print('Performance records inserted:', len(df))

Performance records inserted: 660


In [80]:
# verify all reconciled database tables
tables = ['Country', 'League', 'Club', 'Player', 'Performance']

for table in tables:
    cursor.execute(f'SELECT COUNT(*) FROM {table}')
    count = cursor.fetchone()[0]
    print(f'{table}: {count} rows')

Country: 9 rows
League: 18 rows
Club: 480 rows
Player: 888 rows
Performance: 1320 rows


## Step 5 — ETL Process

We extract data from the reconciled database, transform it into
the star schema format by generating surrogate keys and collapsing
the Club, League and Country hierarchy into a single dimension table,
then load it into the data warehouse tables.

In [81]:
# create dim_player table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_player (
        player_sk INTEGER PRIMARY KEY AUTOINCREMENT,
        player_id INTEGER,
        player_name TEXT
    )
''')
conn.commit()
print('dim_player created')

dim_player created


In [82]:
# create dim_club table with collapsed hierarchy
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_club (
        club_sk INTEGER PRIMARY KEY AUTOINCREMENT,
        club_id INTEGER,
        club_code TEXT,
        league_name TEXT,
        country_name TEXT,
        country_id INTEGER
    )
''')
conn.commit()
print('dim_club created')

dim_club created


In [83]:
# create dim_year table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_year (
        year_sk INTEGER PRIMARY KEY AUTOINCREMENT,
        year INTEGER
    )
''')
conn.commit()
print('dim_year created')

dim_year created


In [84]:
# create fact_performance table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_performance (
        performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player_sk INTEGER NOT NULL,
        club_sk INTEGER NOT NULL,
        year_sk INTEGER NOT NULL,
        matches_played INTEGER,
        substitutions INTEGER,
        minutes INTEGER,
        goals INTEGER,
        xg REAL,
        xg_per_avg_match REAL,
        shots INTEGER,
        on_target INTEGER,
        shots_per_avg_match REAL,
        on_target_per_avg_match REAL,
        FOREIGN KEY (player_sk) REFERENCES dim_player(player_sk),
        FOREIGN KEY (club_sk) REFERENCES dim_club(club_sk),
        FOREIGN KEY (year_sk) REFERENCES dim_year(year_sk)
    )
''')
conn.commit()
print('fact_performance created')

fact_performance created


In [85]:
# load dim_player from reconciled database
cursor.execute('SELECT player_id, player_name FROM Player')
players = cursor.fetchall()

for player in players:
    cursor.execute('''
        INSERT INTO dim_player (player_id, player_name)
        VALUES (?, ?)
    ''', (player[0], player[1]))

conn.commit()
print('dim_player loaded:', len(players), 'rows')

dim_player loaded: 888 rows


In [86]:
# load dim_club with collapsed hierarchy
cursor.execute('''
    SELECT c.club_id, c.club_code,
           l.league_name,
           co.country_name, co.country_id
    FROM Club c
    JOIN League l ON c.league_id = l.league_id
    JOIN Country co ON l.country_id = co.country_id
''')
clubs = cursor.fetchall()

for club in clubs:
    cursor.execute('''
        INSERT INTO dim_club (
            club_id, club_code, league_name,
            country_name, country_id
        )
        VALUES (?, ?, ?, ?, ?)
    ''', (club[0], club[1], club[2], club[3], club[4]))

conn.commit()
print('dim_club loaded:', len(clubs), 'rows')

dim_club loaded: 480 rows


In [87]:
# load dim_year from unique years in Performance table
cursor.execute('SELECT DISTINCT year FROM Performance ORDER BY year')
years = cursor.fetchall()

for year in years:
    cursor.execute('''
        INSERT INTO dim_year (year)
        VALUES (?)
    ''', (year[0],))

conn.commit()
print('dim_year loaded:', len(years), 'rows')

dim_year loaded: 5 rows


In [88]:
# extract all performance records
cursor.execute('SELECT * FROM Performance')
performances = cursor.fetchall()
print('Records to load:', len(performances))

Records to load: 1320


In [89]:
# load fact_performance with surrogate keys
for perf in performances:
    player_id = perf[1]
    club_id = perf[2]
    year = perf[3]

    cursor.execute(
        'SELECT player_sk FROM dim_player WHERE player_id = ?',
        (player_id,)
    )
    player_sk = cursor.fetchone()[0]

    cursor.execute(
        'SELECT club_sk FROM dim_club WHERE club_id = ?',
        (club_id,)
    )
    club_sk = cursor.fetchone()[0]

    cursor.execute(
        'SELECT year_sk FROM dim_year WHERE year = ?',
        (year,)
    )
    year_sk = cursor.fetchone()[0]

    cursor.execute('''
        INSERT INTO fact_performance (
            player_sk, club_sk, year_sk,
            matches_played, substitutions, minutes,
            goals, xg, xg_per_avg_match, shots,
            on_target, shots_per_avg_match,
            on_target_per_avg_match
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        player_sk, club_sk, year_sk,
        perf[4], perf[5], perf[6], perf[7],
        perf[8], perf[9], perf[10], perf[11],
        perf[12], perf[13]
    ))

conn.commit()
print('fact_performance loaded:', len(performances), 'rows')

fact_performance loaded: 1320 rows


In [90]:
# final verification of all data warehouse tables
dw_tables = [
    'dim_player', 'dim_club',
    'dim_year', 'fact_performance'
]

for table in dw_tables:
    cursor.execute(f'SELECT COUNT(*) FROM {table}')
    count = cursor.fetchone()[0]
    print(f'{table}: {count} rows')

dim_player: 1332 rows
dim_club: 720 rows
dim_year: 10 rows
fact_performance: 1980 rows


In [91]:
# make sure all changes are saved to the database file
conn.commit()
print('Database saved successfully')

Database saved successfully


In [92]:
# check the database file was created
import os
print(os.path.exists('football_dw.db'))
print('File size:', os.path.getsize('football_dw.db'), 'bytes')

True
File size: 352256 bytes


## Step 6 — Export Data Warehouse to CSV for Tableau

Tableau Public does not accept direct database connections so
we export the data warehouse tables to CSV files for use in Tableau.

In [93]:
# export dim_player to csv
cursor.execute('SELECT * FROM dim_player')
dim_player_df = pd.DataFrame(
    cursor.fetchall(),
    columns=['player_sk', 'player_id', 'player_name']
)
dim_player_df.to_csv('dim_player.csv', index=False)
print('dim_player exported:', len(dim_player_df), 'rows')

dim_player exported: 1332 rows


In [94]:
# export dim_club to csv
cursor.execute('SELECT * FROM dim_club')
dim_club_df = pd.DataFrame(
    cursor.fetchall(),
    columns=['club_sk', 'club_id', 'club_code',
             'league_name', 'country_name', 'country_id']
)
dim_club_df.to_csv('dim_club.csv', index=False)
print('dim_club exported:', len(dim_club_df), 'rows')

dim_club exported: 720 rows


In [95]:
# export dim_year to csv
cursor.execute('SELECT * FROM dim_year')
dim_year_df = pd.DataFrame(
    cursor.fetchall(),
    columns=['year_sk', 'year']
)
dim_year_df.to_csv('dim_year.csv', index=False)
print('dim_year exported:', len(dim_year_df), 'rows')

dim_year exported: 10 rows


In [96]:
# export fact_performance to csv
cursor.execute('SELECT * FROM fact_performance')
fact_df = pd.DataFrame(
    cursor.fetchall(),
    columns=[
        'performance_id', 'player_sk', 'club_sk', 'year_sk',
        'matches_played', 'substitutions', 'minutes', 'goals',
        'xg', 'xg_per_avg_match', 'shots', 'on_target',
        'shots_per_avg_match', 'on_target_per_avg_match'
    ]
)
fact_df.to_csv('fact_performance.csv', index=False)
print('fact_performance exported:', len(fact_df), 'rows')

fact_performance exported: 1980 rows


In [97]:
# create one combined flat file for easier Tableau use
combined = fact_df.merge(dim_player_df, on='player_sk')
combined = combined.merge(dim_club_df, on='club_sk')
combined = combined.merge(dim_year_df, on='year_sk')
combined.to_csv('football_warehouse.csv', index=False)
print('Combined file exported:', len(combined), 'rows')
print('Columns:', combined.columns.tolist())

Combined file exported: 1980 rows
Columns: ['performance_id', 'player_sk', 'club_sk', 'year_sk', 'matches_played', 'substitutions', 'minutes', 'goals', 'xg', 'xg_per_avg_match', 'shots', 'on_target', 'shots_per_avg_match', 'on_target_per_avg_match', 'player_id', 'player_name', 'club_id', 'club_code', 'league_name', 'country_name', 'country_id', 'year']
